In [5]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress
from itertools import cycle

def calculate_youngs_modulus(file_path):
    data = np.loadtxt(file_path, skiprows=1)
    strain = data[:, 0]
    stress = data[:, 1]  # Using stress in x direction
    slope, _, _, _, _ = linregress(strain, stress)
    return slope

def extract_info_from_path(path):
    parts = path.split(os.path.sep)
    alloy_orientation = parts[-3]
    
    # Handle cases where alloy and orientation might be combined
    if '_' in alloy_orientation:
        alloy, orientation = alloy_orientation.split('_', 1)
    else:
        alloy = alloy_orientation
        orientation = 'unknown'
    
    var_temp_part = parts[-2]
    var_parts = var_temp_part.split('_')
    
    if len(var_parts) == 3:
        var_num, total_atoms, temperature = var_parts
    elif len(var_parts) == 2:
        var_num, temperature = var_parts
        total_atoms = 'unknown'
    else:
        print(f"Unexpected format in {var_temp_part}")
        return None
    
    temperature = int(temperature[:-1])  # Remove 'k' and convert to int
    return alloy, orientation, temperature, var_num, total_atoms

def plot_youngs_modulus_vs_temperature(base_path):
    data = {}
    for root, dirs, files in os.walk(base_path):
        for file in files:
            if file.startswith("SS_curve") and file.endswith(".txt"):
                file_path = os.path.join(root, file)
                info = extract_info_from_path(file_path)
                if info is None:
                    continue
                alloy, orientation, temperature, var_num, total_atoms = info
                youngs_modulus = calculate_youngs_modulus(file_path)
                
                key = (alloy, orientation, var_num)
                if key not in data:
                    data[key] = []
                data[key].append((temperature, youngs_modulus))
    
    # Mapping from original alloy-variant combinations to candidate labels
    candidate_labels = {
        ('NiCoTiZrHf', 'var4'): 'candidate 1',
        ('NiCoTiZrHf', 'var6'): 'candidate 2',
        ('NiCoTiZrHf', 'var8'): 'candidate 3',
        ('NiCoTiZrHf', 'var9'): 'candidate 4',
        ('NiCoTiZr', 'var12'): 'candidate 5',
        ('NiCoTiZr', 'var13'): 'candidate 6'
    }
    
    # Sorting keys by the var number
    sorted_keys = sorted(data.keys(), key=lambda x: int(x[2][3:]))

    # Assigning new colors using a standard color cycle
    unique_alloy_var_pairs = set((key[0], key[2]) for key in sorted_keys)
    color_cycle = cycle(plt.get_cmap('tab10').colors)
    color_dict = {pair: next(color_cycle) for pair in unique_alloy_var_pairs}

    # Plotting
    orientations = set(key[1] for key in data.keys())
    
    for orientation in orientations:
        plt.figure(figsize=(10, 6))
        for key in sorted_keys:
            if key[1] == orientation:
                values = data[key]
                temperatures, moduli = zip(*sorted(values))
                color = color_dict[(key[0], key[2])]
                candidate_label = candidate_labels.get((key[0], key[2]), 'Unknown Candidate')
                plt.plot(temperatures, moduli, marker='o', label=candidate_label, color=color)
        
        plt.xlabel('Temperature (K)')
        plt.ylabel("Young's Modulus (GPa)")
        plt.title(f"Young's Modulus vs Temperature - Orientation {orientation}")
        plt.legend()
        plt.grid(True)
        plt.show()  # Display the plot directly

# Usage: Set the base path to one directory up and then to the "compress" directory
base_path = os.path.join(os.pardir, "compress")
plot_youngs_modulus_vs_temperature(base_path)
